# GPU Notebook — Arithmetic (Addition) Interpretability

Companion to `run_gpu.ipynb`, targeting the **addition** behaviour (carry logic).

**Key difference:** the SAEs were trained on *pooled* activations from all three
behaviours (capitals + addition + units), so they already cover arithmetic. This
notebook **reuses the existing SAE checkpoints** and therefore **skips activation
capture and SAE training entirely** — the ~50 min/prompt training cost is not
repeated here.

### What to expect
Baseline analysis shows the model is **~98% confident on every addition prompt**
(even the hardest carry cases). That means *ablation* effects will likely be small
(there is little uncertainty to remove — the same reason the Kabul capitals prompt
showed nothing). The **swap** experiments (carry ↔ no-carry) are the most likely to
produce a visible, interpretable shift. Prompts here are the *least*-confident carry
cases, chosen to maximise whatever headroom exists.

**Runtime settings:** GPU (any tier), High-RAM recommended.


## Step 1 — Mount Drive & extract project

In [ ]:
# Step 1: Mount Google Drive and extract project
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

# Set to True if you uploaded project.zip directly to the Colab files sidebar:
uploaded_directly_to_colab = False

if uploaded_directly_to_colab:
    zip_path = "/content/project.zip"
else:
    zip_path = "/content/drive/MyDrive/mphil-project/project.zip"

if os.path.exists(zip_path):
    print(f"Extracting {zip_path}...")
    !unzip -q -o {zip_path} -d /content/
    os.chdir('/content')
    print(f"Current working directory: {os.getcwd()}")
    !ls -l
else:
    print(f"ERROR: Could not find {zip_path}.")
    print("Please upload 'project.zip' to Google Drive or the Colab files sidebar.")

## Step 2 — Install dependencies

In [ ]:
# Step 2: Install Dependencies
!pip install --upgrade "transformers>=4.51.0" accelerate
!pip install -e .

## Step 3 — Generate the addition dataset
Only the arithmetic CSV is needed here.

In [ ]:
# Step 3: Generate Datasets (addition CSV is what we need)
!python data/generate_datasets.py
print("\nDataset files:")
!ls -lh data/addition_data.csv

## Step 4 — Restore SAE checkpoints from Drive (no training)

The SAEs are behaviour-agnostic (trained on pooled activations), so we reuse the
checkpoints produced by `run_gpu.ipynb`. This cell copies them back from Drive into
the local workspace. **If they are missing**, run `run_gpu.ipynb` Step 5 first (or
uncomment the training fallback below).

In [ ]:
# Step 4: Restore SAE checkpoints from Drive
import os, shutil, glob

drive_sae = '/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints'
local_sae = '/content/mechanistic_data/sae_checkpoints'
os.makedirs(local_sae, exist_ok=True)

if os.path.isdir(drive_sae) and glob.glob(f'{drive_sae}/sae_layer*.pt'):
    for f in os.listdir(drive_sae):
        shutil.copy2(f'{drive_sae}/{f}', local_sae)
    print("Restored SAE checkpoints from Drive:")
    !ls -lh {local_sae}
else:
    print("No SAE checkpoints found in Drive at:", drive_sae)
    print("Run run_gpu.ipynb Step 5 (training) first, or uncomment the fallback below.")
    # --- Training fallback (slow): captures activations then trains all 8 layers ---
    # !python src/train.py --config configs/sae_config.yaml \
    #   --drive-dir /content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints

---
## Attribution graphs — carry-arithmetic prompts

We use the three least-confident carry cases from the baseline (`58+83=141`,
`76+98=174`, `98+79=177`). The answer tokens are single tokens, so `--target` is
the number string. `attribution_graph.py` prints a ready-to-paste `--features`
string; we instead pass `--graph-json` to the interventions so all graph features
are used automatically.

In [ ]:
# Graph 1: 58 + 83 = 141  (carry)
!python src/attribution_graph.py \
  --prompt "Question: What is 58 + 83? Answer:" \
  --target "141" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/add_58_83_graph.json \
  --output-html outputs/add_58_83_graph.html \
  --output-mermaid outputs/add_58_83_graph.md

In [ ]:
# Graph 2: 76 + 98 = 174  (carry)
!python src/attribution_graph.py \
  --prompt "Question: What is 76 + 98? Answer:" \
  --target "174" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/add_76_98_graph.json \
  --output-html outputs/add_76_98_graph.html \
  --output-mermaid outputs/add_76_98_graph.md

In [ ]:
# Graph 3: 98 + 79 = 177  (carry)
!python src/attribution_graph.py \
  --prompt "Question: What is 98 + 79? Answer:" \
  --target "177" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/add_98_79_graph.json \
  --output-html outputs/add_98_79_graph.html \
  --output-mermaid outputs/add_98_79_graph.md

### Checkpoint: copy graphs to Drive

In [ ]:
# Persist attribution graphs immediately (crash-safe)
import os, shutil, glob
drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for f in glob.glob('/content/outputs/add_*graph*'):
    shutil.copy2(f, drive_out)
    print("Copied", os.path.basename(f))

---
## Diagnostic 1 — Full MLP knockout

Upper bound on how much the hooked layers matter. Given ~98% baseline confidence,
expect this to move the prediction only a little (unlike the low-confidence capitals
prompts). If it does *nothing*, the addition computation isn't in these MLP outputs
at the last token.

In [ ]:
# Full knockout: 58 + 83
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer:" \
  --target-token "141" \
  --full-knockout \
  --output outputs/add_knockout_58_83.json

## Diagnostic 2 — Progressive ablation scan
Ablate the top-10/25/50/100/all graph features and watch the logit of the answer.

In [ ]:
# Scan: 58 + 83
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer:" \
  --target-token "141" \
  --graph-json outputs/add_58_83_graph.json \
  --scan \
  --output outputs/add_scan_58_83.json

## Experiment 1 — Targeted inhibition (all graph features)

In [ ]:
# Inhibit all graph features: 76 + 98
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 76 + 98? Answer:" \
  --target-token "174" \
  --graph-json outputs/add_76_98_graph.json \
  --output outputs/add_inhibit_76_98.json

## Experiment 2 — Activation swap-in (the promising one)

Swap features from a **no-carry** source (`32 + 42 = 74`) into a **carry** target
(`58 + 83 = 141`). If a carry-related circuit exists, injecting no-carry activations
should pull the answer's ones/tens digit or reduce the carry. We track both the
correct answer and the no-carry-style "dropped-carry" distractor (`131`, i.e. the
answer minus the 10s carry) to see if the swap suppresses carrying.

In [ ]:
# Swap: no-carry (32+42=74) source -> carry (58+83=141) target
# Track 141 (correct, with carry) and 131 (carry dropped)
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 32 + 42? Answer:" \
  --prompt "Question: What is 58 + 83? Answer:" \
  --graph-json outputs/add_58_83_graph.json \
  --target-token "141, 131" \
  --output outputs/add_swap_nocarry_to_carry.json

In [ ]:
# Swap: full latent swap (no --features / --graph-json) — swaps everything
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 32 + 42? Answer:" \
  --prompt "Question: What is 58 + 83? Answer:" \
  --target-token "141, 131, 74" \
  --output outputs/add_swap_full_nocarry_to_carry.json

---
## Copy all outputs to Drive

In [ ]:
# Copy all addition outputs to Google Drive for persistence
import os, shutil, glob
drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for src in glob.glob('/content/outputs/add_*'):
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print('Copied', os.path.basename(src))
print('Done!')

---
## Notes on interpreting these results

- **Low ablation effect is an expected result, not a bug.** The model is ~98%
  confident on every addition prompt, so there is little logit mass for ablation to
  move. Report the knockout/scan deltas as evidence of *where* (or whether) the
  arithmetic is computed in these MLP layers.
- **The swap is the informative experiment.** A shift toward the dropped-carry
  distractor (`131`) under a no-carry swap would be direct evidence of a
  carry-handling feature.
- **Layer 32's SAE is degenerate** (val_mse ≈ 0.12 vs ≈ 0.002 at layer 4). If a
  result hinges on layer-32 features, treat it skeptically or drop layer 32 via
  `--layers 4 8 12 16 20 24 28`.
